# YGO Baseline v2 - Single Model (8-class Rarity) + Derived Foil Tasks

Approach mới theo yêu cầu:
- **Task 1**: train **1 model duy nhất** dự đoán 8 rarity classes.
- Dùng 3 input crops: **name / art / full** với 3 augmentation pipeline khác nhau.
- **Task 2/3/4**: suy ra trực tiếp từ rarity prediction theo bảng mapping.

Pipeline: `train.csv -> 3-crop dataset -> 1 tri-branch model -> rarity logits(8) -> rarity pred -> derive name_foil/art_foil/full_foil`.

In [ ]:
# !pip install -q timm albumentations opencv-python-headless scikit-learn tqdm

In [ ]:

import os
import random
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


In [ ]:

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


In [ ]:

TRAIN_CSV = 'data/train.csv'
TRAIN_IMG_DIR = 'data/train/images'
TEST_IMG_DIR = 'data/test/images'
SAMPLE_SUB = 'data/sample_submission.csv'
OUTPUT_DIR = 'outputs_baseline_single_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RARITY_TO_ID = {
    'Common': 1,
    'Rare': 2,
    'Super Rare': 3,
    'Ultra Rare': 4,
    'Secret Rare': 5,
    'Prismatic Secret Rare': 6,
    'Quarter Century Secret Rare': 7,
    'Starlight Rare': 8,
}
ID_TO_RARITY = {v:k for k,v in RARITY_TO_ID.items()}

# Derive Task 2/3/4 from rarity
RARITY_TO_FOIL = {
    1: (0,0,0),
    2: (1,0,0),
    3: (0,1,0),
    4: (1,1,0),
    5: (1,1,0),
    6: (1,1,0),
    7: (1,1,1),
    8: (1,1,1),
}


## 1) Crop config + 3 augmentation pipelines

In [ ]:

CROP_CFG = {
    'name': dict(x1=0.06, y1=0.04, x2=0.94, y2=0.13),
    'art':  dict(x1=0.10, y1=0.17, x2=0.90, y2=0.56),
    'full': dict(x1=0.00, y1=0.00, x2=1.00, y2=1.00),
}

def crop_by_region(img, region):
    h, w = img.shape[:2]
    c = CROP_CFG[region]
    x1, y1 = int(c['x1']*w), int(c['y1']*h)
    x2, y2 = int(c['x2']*w), int(c['y2']*h)
    return img[y1:y2, x1:x2]


def get_transforms(train=True):
    # 3 transform riêng để học tín hiệu khác nhau
    name_train = A.Compose([
        A.Resize(96, 384),
        A.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.25, hue=0.03, p=0.8),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
        A.GaussNoise(std_range=(0.01, 0.03), p=0.25),
        A.Normalize(), ToTensorV2(),
    ])
    art_train = A.Compose([
        A.Resize(256, 256),
        A.ColorJitter(brightness=0.15, contrast=0.20, saturation=0.20, hue=0.02, p=0.8),
        A.GaussianBlur(blur_limit=(3,5), p=0.2),
        A.Sharpen(alpha=(0.1, 0.3), p=0.2),
        A.GaussNoise(std_range=(0.01, 0.03), p=0.2),
        A.Normalize(), ToTensorV2(),
    ])
    full_train = A.Compose([
        A.Resize(384, 384),
        A.ColorJitter(brightness=0.12, contrast=0.15, saturation=0.15, hue=0.02, p=0.7),
        A.RandomGamma(gamma_limit=(85,115), p=0.4),
        A.GaussNoise(std_range=(0.01,0.02), p=0.2),
        A.Normalize(), ToTensorV2(),
    ])

    name_val = A.Compose([A.Resize(96, 384), A.Normalize(), ToTensorV2()])
    art_val = A.Compose([A.Resize(256, 256), A.Normalize(), ToTensorV2()])
    full_val = A.Compose([A.Resize(384, 384), A.Normalize(), ToTensorV2()])

    if train:
        return {'name': name_train, 'art': art_train, 'full': full_train}
    return {'name': name_val, 'art': art_val, 'full': full_val}


## 2) Tri-crop dataset

In [ ]:

class CardTriCropDataset(Dataset):
    def __init__(self, df, img_dir, transforms, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, row['id'])
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        name = crop_by_region(img, 'name')
        art  = crop_by_region(img, 'art')
        full = crop_by_region(img, 'full')

        name = self.transforms['name'](image=name)['image']
        art  = self.transforms['art'](image=art)['image']
        full = self.transforms['full'](image=full)['image']

        if self.is_test:
            return name, art, full, row['id']

        y = int(row['rarity']) - 1  # 0..7 for CE
        return name, art, full, torch.tensor(y, dtype=torch.long)


## 3) Single model: 3 branches -> fusion -> 8-class logits

In [ ]:

class TriCropRarityModel(nn.Module):
    def __init__(self, backbone='resnet18', pretrained=True, n_classes=8):
        super().__init__()
        self.name_encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.art_encoder  = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.full_encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')

        feat_dim = self.name_encoder.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 3, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, n_classes)
        )

    def forward(self, x_name, x_art, x_full):
        f1 = self.name_encoder(x_name)
        f2 = self.art_encoder(x_art)
        f3 = self.full_encoder(x_full)
        f = torch.cat([f1, f2, f3], dim=1)
        return self.head(f)


## 4) Train (Task1 only) + metric

In [ ]:

@dataclass
class CFG:
    n_splits: int = 5
    epochs: int = 10
    batch_size: int = 12
    lr: float = 3e-4
    wd: float = 1e-4
    num_workers: int = 2

cfg = CFG()

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')


def train_fold(train_df, fold, tr_idx, va_idx):
    tr_df = train_df.iloc[tr_idx].copy()
    va_df = train_df.iloc[va_idx].copy()

    tr_ds = CardTriCropDataset(tr_df, TRAIN_IMG_DIR, get_transforms(train=True), is_test=False)
    va_ds = CardTriCropDataset(va_df, TRAIN_IMG_DIR, get_transforms(train=False), is_test=False)

    tr_dl = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    va_dl = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

    model = TriCropRarityModel(backbone='resnet18', pretrained=True, n_classes=8).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    best_f1 = -1
    best_path = os.path.join(OUTPUT_DIR, f'rarity_fold{fold}.pth')

    for epoch in range(cfg.epochs):
        model.train()
        for n,a,f,y in tr_dl:
            n,a,f,y = n.to(DEVICE), a.to(DEVICE), f.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(n,a,f)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        model.eval()
        pred_cls, true_cls = [], []
        with torch.no_grad():
            for n,a,f,y in va_dl:
                n,a,f = n.to(DEVICE), a.to(DEVICE), f.to(DEVICE)
                logits = model(n,a,f)
                p = torch.argmax(logits, dim=1).cpu().numpy()
                pred_cls.extend(p.tolist())
                true_cls.extend(y.numpy().tolist())

        f1 = macro_f1(true_cls, pred_cls)
        print(f'[fold {fold}] epoch {epoch}: rarity_macro_f1={f1:.4f}')
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), best_path)

    return best_path


In [ ]:

train_df = pd.read_csv(TRAIN_CSV)
if train_df['rarity'].dtype == object:
    train_df['rarity'] = train_df['rarity'].map(RARITY_TO_ID)

skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=SEED)
model_paths = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['rarity'])):
    p = train_fold(train_df, fold, tr_idx, va_idx)
    model_paths.append(p)

print('Saved models:', model_paths)


## 5) Inference Task1 -> derive Task2/3/4

In [ ]:

def load_models(paths):
    models = []
    for p in paths:
        m = TriCropRarityModel(backbone='resnet18', pretrained=False, n_classes=8).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models.append(m)
    return models


def predict_rarity(test_df, model_paths):
    ds = CardTriCropDataset(test_df, TEST_IMG_DIR, get_transforms(train=False), is_test=True)
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    models = load_models(model_paths)

    all_ids, all_pred = [], []
    with torch.no_grad():
        for n,a,f,ids in tqdm(dl, desc='predict rarity'):
            n,a,f = n.to(DEVICE), a.to(DEVICE), f.to(DEVICE)
            probs = []
            for m in models:
                probs.append(torch.softmax(m(n,a,f), dim=1).cpu().numpy())
            p = np.mean(probs, axis=0)
            y = np.argmax(p, axis=1) + 1  # back to 1..8
            all_ids.extend(ids)
            all_pred.extend(y.tolist())
    return pd.DataFrame({'id': all_ids, 'rarity': all_pred})

sample = pd.read_csv(SAMPLE_SUB)
test_df = sample[['id']].copy()
pred = predict_rarity(test_df, model_paths)

# derive task 2/3/4 from rarity
pred['name_foil'] = pred['rarity'].map(lambda r: RARITY_TO_FOIL[int(r)][0])
pred['art_foil']  = pred['rarity'].map(lambda r: RARITY_TO_FOIL[int(r)][1])
pred['full_foil'] = pred['rarity'].map(lambda r: RARITY_TO_FOIL[int(r)][2])

sub = pred[['id','rarity','name_foil','art_foil','full_foil']].copy()
sub.to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)
sub.head()


## 6) Notes

- Nếu muốn mạnh hơn: thay backbone `convnext_tiny` hoặc `efficientnet_b3`.
- Có thể thêm weighted CE/focal loss nếu mất cân bằng class.
- Vì task 2/3/4 là hàm xác định của rarity trong đề, cách này đảm bảo consistency 100%.